# 🌳 AdaBoost vs XGBoost — Kapsamlı Karşılaştırma

---

## 📚 1. Boosting Nedir?

**Boosting**, zayıf öğrenicileri (weak learners) sıralı olarak birleştirip güçlü bir model oluşturan bir **topluluk öğrenme (ensemble learning)** yöntemidir.

- Her yeni model, bir öncekinin **hata yaptığı örneklere** daha fazla odaklanır.
- Modeller sıralı (sequential) çalışır, paralel değil.
- Bias'ı azaltmak için idealdir.

```
Zayıf Model 1 → Zayıf Model 2 → Zayıf Model 3 → ... → Güçlü Model
     ↑ hataları              ↑ hataları
     düzelt                 düzelt
```

---

## 🔴 2. AdaBoost (Adaptive Boosting)

### Ne Yapar?
- **1995'te** Freund & Schapire tarafından geliştirildi.
- Her iterasyonda **yanlış sınıflandırılan örneklerin ağırlığını artırır**.
- Her zayıf öğrenici (genellikle 1 katmanlı karar ağacı = **stump**) bir ağırlık alır.
- Final tahmin, ağırlıklı oylamadır.

### Matematiksel Temel

**Adım 1:** Her örnek eşit ağırlıkla başlar: $w_i = \frac{1}{N}$

**Adım 2:** Hata hesapla:
$$\epsilon_t = \sum_{i: h_t(x_i) \neq y_i} w_i$$

**Adım 3:** Model ağırlığı:
$$\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$$

**Adım 4:** Ağırlıkları güncelle:
$$w_i \leftarrow w_i \cdot e^{-\alpha_t y_i h_t(x_i)}$$

**Final Tahmin:**
$$H(x) = \text{sign}\left(\sum_t \alpha_t h_t(x)\right)$$

### Özellikleri
| Özellik | Değer |
|---------|-------|
| Zayıf öğrenici | Decision Stump (depth=1) |
| Hata düzeltme | Örnek ağırlıkları |
| Regularizasyon | Yok (overfitting riski) |
| Hız | Hızlı |
| Outlier hassasiyeti | **Yüksek** |

---

## 🟢 3. XGBoost (Extreme Gradient Boosting)

### Ne Yapar?
- **2016'da** Tianqi Chen tarafından geliştirildi.
- **Gradient Boosting** üzerine kurulu, ancak çok daha optimize.
- Her model, öncekinin **rezidüel hatalarını** (gradient) öğrenir.
- L1 + L2 regularizasyon, paralel işlem, eksik veri desteği içerir.

### Matematiksel Temel

**Objektif Fonksiyon:**
$$\mathcal{L} = \sum_i l(y_i, \hat{y}_i) + \sum_k \Omega(f_k)$$

**Regularizasyon Terimi:**
$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \|w\|^2$$

**2. Derece Taylor Açılımı (hızlandırma için):**
$$\mathcal{L}^{(t)} \approx \sum_i \left[g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)\right] + \Omega(f_t)$$

- $g_i$: 1. türev (gradient)
- $h_i$: 2. türev (hessian)

### Özellikleri
| Özellik | Değer |
|---------|-------|
| Zayıf öğrenici | Derin karar ağaçları |
| Hata düzeltme | Gradient (artık hatalar) |
| Regularizasyon | **L1 + L2** |
| Hız | **Çok Hızlı** (paralel) |
| Eksik veri | **Otomatik işler** |
| Outlier hassasiyeti | Düşük |

---

## 📊 4. AdaBoost vs XGBoost — Özet Karşılaştırma

| Kriter | AdaBoost | XGBoost |
|--------|----------|----------|
| Yıl | 1995 | 2016 |
| Temel Fikir | Örnek ağırlıklandırma | Gradient descent |
| Ağaç derinliği | 1 (stump) | Ayarlanabilir |
| Regularizasyon | ❌ | ✅ L1+L2 |
| Eksik veri | ❌ | ✅ |
| Paralel hesap | ❌ | ✅ |
| Outlier duyarlılığı | Yüksek | Düşük |
| Hiperparametre | Az | Çok |
| Performans | İyi | **Çok İyi** |

---
## 💻 5. Kütüphaneler ve Veri Hazırlığı

In [ ]:
# ─── Temel veri işleme kütüphaneleri ───────────────────────────────────────
import numpy as np          # Sayısal hesaplamalar (dizi işlemleri, lineer cebir)
import pandas as pd         # Tablo (DataFrame) yapısında veri işleme
import matplotlib.pyplot as plt  # Grafik çizimi
import seaborn as sns       # matplotlib üzerine kurulu, daha şık görselleştirme
import warnings
warnings.filterwarnings('ignore')  # Gereksiz uyarıları gizle

# ─── Scikit-learn araçları ──────────────────────────────────────────────────
from sklearn.datasets import make_classification, load_breast_cancer
# load_breast_cancer : 569 örnek, 30 özellik içeren gerçek tıbbi veri seti

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
# train_test_split : veriyi eğitim / test olarak böler
# cross_val_score  : k-fold çapraz doğrulama skoru
# learning_curve   : farklı eğitim boyutlarında performans eğrisi

from sklearn.ensemble import AdaBoostClassifier
# AdaBoost sınıflandırıcı — sklearn içinde hazır gelir

from sklearn.tree import DecisionTreeClassifier
# AdaBoost'un zayıf öğrenicisi olarak kullanılacak karar ağacı

from sklearn.preprocessing import StandardScaler
# Özellikleri ortalaması 0, std'si 1 olacak şekilde ölçekler

from sklearn.metrics import (
    accuracy_score,         # Doğru tahmin oranı
    classification_report,  # Precision / Recall / F1 özet tablosu
    confusion_matrix,       # Hangi sınıf hangisiyle karıştırıldı?
    roc_auc_score,          # ROC eğrisinin altındaki alan (0.5=rastgele, 1=mükemmel)
    roc_curve,              # FPR ve TPR değerleri (ROC grafiği için)
    f1_score,               # Precision ve Recall'ın harmonik ortalaması
    precision_score,        # TP / (TP + FP) — "Tahmin ettiklerimin kaçı doğru?"
    recall_score            # TP / (TP + FN) — "Gerçeklerin kaçını yakaladım?"
)

import xgboost as xgb      # XGBoost kütüphanesi (pip install xgboost)
import time                 # Eğitim süresi ölçümü için

# ─── Görsel ayarlar ─────────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)      # Varsayılan grafik boyutu
plt.rcParams['font.size'] = 12                # Varsayılan yazı boyutu
plt.style.use('seaborn-v0_8-whitegrid')       # Beyaz ızgara temalı stil

print('✅ Tüm kütüphaneler başarıyla yüklendi!')
print(f'   XGBoost versiyonu: {xgb.__version__}')

---
## 🗂️ 6. Veri Seti — Breast Cancer (Meme Kanseri)

In [ ]:
# ─── Veri setini yükle ──────────────────────────────────────────────────────
data = load_breast_cancer()
# data.data   : (569, 30) boyutlu numpy dizisi — sayısal özellikler
# data.target : (569,) boyutlu numpy dizisi — 0=Malignant, 1=Benign

# Pandas DataFrame'e çevir (sütun isimlerini de ekle)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='hedef')

print('📋 Veri Seti Bilgileri')
print('='*40)
print(f'Özellik sayısı  : {X.shape[1]}')
print(f'Örnek sayısı    : {X.shape[0]}')
print(f'Sınıf dağılımı  : {dict(y.value_counts())}')
print(f'  0 = Malignant (Kötü huylu): {sum(y==0)}')
print(f'  1 = Benign (İyi huylu)    : {sum(y==1)}')
print()

# ─── Eğitim / Test olarak böl ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # %20 test, %80 eğitim
    random_state=42,    # Tekrar üretilebilirlik için sabit tohum
    stratify=y          # Her iki sette de sınıf oranı korunur (dengeli bölme)
)

print(f'Eğitim seti: {X_train.shape[0]} örnek')
print(f'Test seti  : {X_test.shape[0]} örnek')

X.head()  # İlk 5 satırı göster

In [ ]:
# ─── Veri setini görselleştir ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol grafik: sınıf dağılımını pasta grafikle göster
counts = y.value_counts()
labels = ['Malignant (Kötü)', 'Benign (İyi)']
colors = ['#e74c3c', '#2ecc71']
axes[0].pie(
    counts,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',       # Yüzde değerlerini göster
    startangle=90,           # 90 dereceden başla
    shadow=True,             # Gölge efekti
    textprops={'fontsize': 12}
)
axes[0].set_title('Sınıf Dağılımı', fontsize=14, fontweight='bold')

# Sağ grafik: ilk özelliğin iki sınıfa göre histogramı
# → İki sınıf arasında bu özelliğin nasıl farklılaştığını gösterir
features_to_plot = data.feature_names[:5]
for i, feat in enumerate(features_to_plot):
    axes[1].hist(X[feat][y==0], alpha=0.5, label='Malignant', color='#e74c3c', bins=20)
    axes[1].hist(X[feat][y==1], alpha=0.5, label='Benign',    color='#2ecc71', bins=20)
    break  # Döngüyü ilk özellikte durdur; sadece 1 özellik yeterli
axes[1].set_title(f'Dağılım: {features_to_plot[0]}', fontsize=14, fontweight='bold')
axes[1].set_xlabel(features_to_plot[0])
axes[1].legend()

plt.tight_layout()
plt.suptitle('Veri Seti Analizi', fontsize=16, fontweight='bold', y=1.02)
plt.show()

---
## 🔴 7. AdaBoost Modeli

In [ ]:
print('🔴 ADABOOST MODELİ EĞİTİLİYOR...')
print('='*45)

# ─── AdaBoost modelini tanımla ──────────────────────────────────────────────
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    # Zayıf öğrenici olarak 'Decision Stump' kullanıyoruz.
    # max_depth=1 → yalnızca tek bir karar dalı; kasıtlı olarak zayıf tutulur.
    # Güçlü bir ağaç seçmek AdaBoost'un dengeleme mantığını bozar.

    n_estimators=100,
    # Kaç tane zayıf öğrenici oluşturulacak?
    # Artırmak genellikle doğruluğu iyileştirir ama bir noktadan sonra plateau oluşur.

    learning_rate=0.1,
    # Her zayıf öğrenicinin final tahmine katkısını küçülten çarpan.
    # Küçük değer → daha yavaş öğrenme ama daha iyi genelleme.
    # learning_rate ile n_estimators arasında ters orantı vardır.

    algorithm='SAMME',
    # 'SAMME'  : sınıf etiketleri (hard vote) kullanır — sklearn>=1.2'de varsayılan.
    # 'SAMME.R': olasılık tahmini kullanır, genellikle daha iyi ama yavaş.

    random_state=42         # Tekrar üretilebilir sonuçlar için sabit tohum
)

# ─── Eğitim ─────────────────────────────────────────────────────────────────
start = time.time()                         # Saat başlatılıyor
ada_model.fit(X_train, y_train)             # Modeli eğitim verisiyle öğret
ada_time = time.time() - start              # Geçen süreyi hesapla

# ─── Tahmin ─────────────────────────────────────────────────────────────────
ada_pred = ada_model.predict(X_test)
# predict() → her örnek için sınıf etiketi döner (0 veya 1)

ada_prob = ada_model.predict_proba(X_test)[:, 1]
# predict_proba() → her örnek için [P(sınıf=0), P(sınıf=1)] döner.
# [:, 1] → sadece pozitif sınıf (Benign) olasılığını al → ROC için gerekli

# ─── Performans metrikleri ───────────────────────────────────────────────────
ada_acc    = accuracy_score(y_test, ada_pred)
# Doğruluk = doğru tahmin sayısı / toplam tahmin sayısı

ada_auc    = roc_auc_score(y_test, ada_prob)
# ROC eğrisinin altındaki alan; 0.5=rastgele, 1.0=mükemmel

ada_f1     = f1_score(y_test, ada_pred)
# F1 = 2 * (Precision * Recall) / (Precision + Recall)
# Dengesiz veri setlerinde accuracy'den daha güvenilir

ada_prec   = precision_score(y_test, ada_pred)
# Precision: "Benign" dediğimizin gerçekten Benign olma oranı

ada_recall = recall_score(y_test, ada_pred)
# Recall (Sensitivity): Gerçek Benign'lerin ne kadarını yakaladık?

print(f'✅ Eğitim Süresi   : {ada_time:.4f} sn')
print(f'📊 Doğruluk (Acc)  : {ada_acc:.4f}  ({ada_acc*100:.2f}%)')
print(f'📊 AUC-ROC         : {ada_auc:.4f}')
print(f'📊 F1 Skoru        : {ada_f1:.4f}')
print(f'📊 Precision       : {ada_prec:.4f}')
print(f'📊 Recall          : {ada_recall:.4f}')
print()
print('📋 Detaylı Rapor:')
print(classification_report(y_test, ada_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# ─── n_estimators'a göre performans analizi ─────────────────────────────────
# Kaç tane zayıf öğrenici kullanılırsa en iyi sonuç alınır?
# 1'den 200'e kadar 10'ar adımla deniyoruz.

n_estimators_range = range(1, 201, 10)
ada_train_scores = []   # Her n için eğitim doğruluğunu saklayacak liste
ada_test_scores  = []   # Her n için test doğruluğunu saklayacak liste

for n in n_estimators_range:
    m = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n,
        learning_rate=0.1,
        algorithm='SAMME',
        random_state=42
    )
    m.fit(X_train, y_train)   # Her n için ayrı model eğit
    ada_train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    ada_test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(12, 5))

# '-o' : noktalı çizgi (o=daire marker, - = düz çizgi)
plt.plot(n_estimators_range, ada_train_scores, 'r-o', markersize=4,
         label='Eğitim Doğruluğu', linewidth=2)

# '--s' : kesik çizgi (s=kare marker, --=kesikli)
plt.plot(n_estimators_range, ada_test_scores, 'r--s', markersize=4,
         label='Test Doğruluğu', linewidth=2)

# Eğitim ve test eğrileri birbirinden uzaklaşıyorsa → OVERFITTING sinyali

plt.xlabel('Zayıf Öğrenici Sayısı (n_estimators)', fontsize=12)
plt.ylabel('Doğruluk', fontsize=12)
plt.title('🔴 AdaBoost — n_estimators vs Doğruluk', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.ylim(0.90, 1.01)
plt.tight_layout()
plt.show()

---
## 🟢 8. XGBoost Modeli

In [ ]:
print('🟢 XGBOOST MODELİ EĞİTİLİYOR...')
print('='*45)

# ─── XGBoost modelini tanımla ───────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    # Toplam ağaç (iterasyon) sayısı.
    # Her iterasyonda bir ağaç eklenir, bir öncekinin hatasını düzeltir.

    max_depth=3,
    # Her ağacın maksimum derinliği.
    # Derinlik arttıkça model karmaşıklaşır → overfitting riski.
    # Genellikle 3-6 arası iyi sonuç verir.

    learning_rate=0.1,
    # Eta (η) olarak da bilinir.
    # Her yeni ağacın katkısını bu katsayıyla küçültür.
    # Küçük lr + fazla ağaç → daha iyi ama daha yavaş.

    subsample=0.8,
    # Her ağacı eğitirken eğitim verisinin %80'ini rastgele seç.
    # Kalan %20 görmezden gelinir → varyansı düşürür (Random Forest'taki gibi).

    colsample_bytree=0.8,
    # Her ağaç için özelliklerin %80'ini rastgele kullan.
    # Özellikler arasındaki korelasyonu kırar, modeli güçlendirir.

    reg_alpha=0.1,
    # L1 (Lasso) regularizasyon katsayısı.
    # Bazı ağırlıkları sıfıra iter → seyrek (sparse) model → özellik seçimi etkisi.

    reg_lambda=1.0,
    # L2 (Ridge) regularizasyon katsayısı.
    # Ağırlıkları küçük tutar → overfitting'i önler.
    # XGBoost'ta varsayılan değer 1.0'dır.

    use_label_encoder=False,
    # Eski XGBoost sürümlerinde etiketleri kodlamak için kullanılırdı.
    # Artık gerekli değil; uyarı mesajını önlemek için False.

    eval_metric='logloss',
    # Eğitim sırasında izlenecek metrik.
    # logloss (log loss): olasılık tahminlerinin kalitesini ölçer.
    # Düştükçe model daha iyi öğreniyor demektir.

    random_state=42,        # Tekrar üretilebilirlik için sabit tohum
    verbosity=0             # 0 = hiç log yazdırma
)

# ─── Eğitim ─────────────────────────────────────────────────────────────────
start = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    # eval_set: her iterasyonda bu veri setlerinde logloss hesapla.
    # validation_0 → eğitim seti, validation_1 → test seti.
    # Sonradan evals_result() ile bu değerlere erişip grafik çizebiliriz.
    verbose=False           # İterasyon başına log yazdırma
)
xgb_time = time.time() - start

# ─── Tahmin ─────────────────────────────────────────────────────────────────
xgb_pred = xgb_model.predict(X_test)
# Sınıf etiketi tahmini: 0 veya 1

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
# Pozitif sınıf (Benign=1) olasılıkları → AUC-ROC hesabı için gerekli

# ─── Performans metrikleri ───────────────────────────────────────────────────
xgb_acc    = accuracy_score(y_test, xgb_pred)
xgb_auc    = roc_auc_score(y_test, xgb_prob)
xgb_f1     = f1_score(y_test, xgb_pred)
xgb_prec   = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)

print(f'✅ Eğitim Süresi   : {xgb_time:.4f} sn')
print(f'📊 Doğruluk (Acc)  : {xgb_acc:.4f}  ({xgb_acc*100:.2f}%)')
print(f'📊 AUC-ROC         : {xgb_auc:.4f}')
print(f'📊 F1 Skoru        : {xgb_f1:.4f}')
print(f'📊 Precision       : {xgb_prec:.4f}')
print(f'📊 Recall          : {xgb_recall:.4f}')
print()
print('📋 Detaylı Rapor:')
print(classification_report(y_test, xgb_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# ─── XGBoost: Özellik Önemi + Eğitim Eğrisi ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Sol: Özellik Önemi ──────────────────────────────────────────────────────
feat_importance = pd.Series(
    xgb_model.feature_importances_,
    # feature_importances_: her özelliğin kaç kez ve ne kadar bilgi kazancıyla
    # bölme noktası olarak seçildiğine dayanan önem skoru (0-1 arası, toplam=1)
    index=data.feature_names
).sort_values(ascending=True).tail(15)
# sort_values(ascending=True): küçükten büyüğe → yatay barda alta koymak için
# .tail(15): sadece en önemli 15 özelliği al

feat_importance.plot(kind='barh', ax=axes[0], color='#2ecc71', edgecolor='white')
axes[0].set_title('🟢 XGBoost — En Önemli 15 Özellik', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Önem Skoru')

# ── Sağ: Eğitim Eğrisi (Log Loss) ──────────────────────────────────────────
results = xgb_model.evals_result()
# evals_result(): fit() sırasında eval_set ile kayıt altına alınan
# her iterasyondaki logloss değerlerini dict olarak döner.
# results['validation_0'] → eğitim seti sonuçları
# results['validation_1'] → test seti sonuçları

axes[1].plot(results['validation_0']['logloss'],
             label='Eğitim', color='#3498db', linewidth=2)
axes[1].plot(results['validation_1']['logloss'],
             label='Test', color='#e74c3c', linewidth=2, linestyle='--')
# Beklenen davranış: her iki eğri de düşmeli.
# Test kaybı düşüp tekrar yükseliyorsa → erken durdurma (early stopping) önerilir.

axes[1].set_title('🟢 XGBoost — Eğitim Eğrisi (Log Loss)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('İterasyon')
axes[1].set_ylabel('Log Loss')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## ⚖️ 9. Kapsamlı Karşılaştırma

In [ ]:
# ─── Confusion Matrix Karşılaştırması ───────────────────────────────────────
# Confusion Matrix: tahmin edilen sınıf vs gerçek sınıf tablosu.
#   - Diagonal (sol-üst, sağ-alt): doğru tahminler (TP ve TN)
#   - Diagonal dışı: yanlış tahminler (FP ve FN)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# zip() ile iki modelin verisi aynı anda işleniyor
for ax, pred, title, color in zip(
    axes,
    [ada_pred, xgb_pred],      # Her modelin tahminleri
    ['🔴 AdaBoost', '🟢 XGBoost'],
    ['Reds', 'Greens']         # Her model için renk teması
):
    cm = confusion_matrix(y_test, pred)
    # [[TN, FP],
    #  [FN, TP]]  formatında 2x2 matris döner

    sns.heatmap(
        cm,
        annot=True,             # Hücrelere sayıları yaz
        fmt='d',                # Tam sayı formatı (float değil)
        cmap=color,             # Renk haritası
        ax=ax,
        xticklabels=['Malignant', 'Benign'],
        yticklabels=['Malignant', 'Benign'],
        linewidths=2,
        linecolor='white',
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    ax.set_title(f'{title}\nConfusion Matrix', fontsize=13, fontweight='bold')
    ax.set_ylabel('Gerçek Sınıf', fontsize=11)
    ax.set_xlabel('Tahmin Edilen Sınıf', fontsize=11)

plt.suptitle('Confusion Matrix Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── ROC Eğrisi + Metrik Bar Grafiği ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Sol: ROC Eğrisi ──────────────────────────────────────────────────────────
# ROC (Receiver Operating Characteristic):
#   X ekseni → FPR (False Positive Rate = FP / (FP + TN))
#   Y ekseni → TPR (True Positive Rate  = TP / (TP + FN))
# Eşik değeri 0'dan 1'e doğru değiştirilerek farklı FPR/TPR çiftleri elde edilir.
# Sol üst köşeye yakın eğri → daha iyi model.

for prob, label, color, acc, auc in [
    (ada_prob, 'AdaBoost', '#e74c3c', ada_acc, ada_auc),
    (xgb_prob, 'XGBoost',  '#2ecc71', xgb_acc, xgb_auc)
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    # roc_curve() → farklı eşik değerleri için (fpr, tpr, thresholds) döner
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5,
                 label=f'{label} (AUC={auc:.4f})')

# Çapraz çizgi: AUC=0.5 → tamamen rastgele tahmin yapan model
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Rastgele Tahmin')
axes[0].fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=12)
axes[0].set_title('ROC Eğrisi Karşılaştırması', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── Sağ: Metrik Bar Grafiği ──────────────────────────────────────────────────
metrics    = ['Accuracy', 'AUC-ROC', 'F1 Score', 'Precision', 'Recall']
ada_scores = [ada_acc, ada_auc, ada_f1, ada_prec, ada_recall]
xgb_scores = [xgb_acc, xgb_auc, xgb_f1, xgb_prec, xgb_recall]

x = np.arange(len(metrics))   # [0, 1, 2, 3, 4] → her metrik için x konumu
width = 0.35                   # Her barın genişliği

# İki model için yan yana bar: biri sola (- width/2), biri sağa (+ width/2)
bars1 = axes[1].bar(x - width/2, ada_scores, width, label='AdaBoost',
                     color='#e74c3c', alpha=0.85, edgecolor='white', linewidth=1.5)
bars2 = axes[1].bar(x + width/2, xgb_scores, width, label='XGBoost',
                     color='#2ecc71', alpha=0.85, edgecolor='white', linewidth=1.5)

# Her barın üstüne sayısal değer yaz
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{bar.get_height():.3f}', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{bar.get_height():.3f}', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics, fontsize=11)
axes[1].set_ylabel('Skor', fontsize=12)
axes[1].set_title('Tüm Metrikler Karşılaştırması', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_ylim(0.88, 1.02)
axes[1].grid(True, axis='y', alpha=0.3)

plt.suptitle('AdaBoost vs XGBoost — Performans Karşılaştırması',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 5-Fold Cross Validation ────────────────────────────────────────────────
# Cross validation neden gerekli?
# Tek train/test bölmesi şansa bağlıdır. CV, veriyi 5 farklı şekilde bölerek
# daha güvenilir ve genellenebilir bir performans tahmini verir.

print('🔄 5-Fold Cross Validation Yapılıyor...')
print('='*45)

ada_cv = cross_val_score(ada_model, X, y, cv=5, scoring='accuracy')
# cv=5 : veriyi 5 parçaya böl, her seferinde 1 parça test, 4 parça eğitim
# scoring='accuracy' : her fold için doğruluk skoru hesapla
# Dönen değer: 5 elemanlı numpy dizisi (her fold için 1 skor)

xgb_cv = cross_val_score(xgb_model, X, y, cv=5, scoring='accuracy')

print(f'🔴 AdaBoost  CV Scores: {ada_cv.round(4)}')
print(f'   Ortalama: {ada_cv.mean():.4f} ± {ada_cv.std():.4f}')
# std küçükse → model farklı veri bölmelerinde tutarlı davranıyor (iyi!)
print()
print(f'🟢 XGBoost   CV Scores: {xgb_cv.round(4)}')
print(f'   Ortalama: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')

# ─── CV görselleştir ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)

ax.plot(x, ada_cv, 'r-o', markersize=8, linewidth=2,
        label=f'AdaBoost (ort={ada_cv.mean():.4f})')
ax.plot(x, xgb_cv, 'g-s', markersize=8, linewidth=2,
        label=f'XGBoost (ort={xgb_cv.mean():.4f})')

# Shaded band: ortalama ± 1 std aralığını gölgeli bant olarak göster
# Bu bant dar ise model tutarlı, geniş ise değişken demektir.
ax.fill_between(x,
                ada_cv.mean() - ada_cv.std(),
                ada_cv.mean() + ada_cv.std(),
                alpha=0.1, color='red')
ax.fill_between(x,
                xgb_cv.mean() - xgb_cv.std(),
                xgb_cv.mean() + xgb_cv.std(),
                alpha=0.1, color='green')

ax.set_xticks(x)
ax.set_xticklabels(folds)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('5-Fold Cross Validation Karşılaştırması', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0.93, 1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Farklı veri boyutlarında eğitim süresi analizi ─────────────────────────
# Veri büyüdükçe AdaBoost mu, XGBoost mu daha hızlı?

sizes = [50, 100, 200, 300, 400, 455]   # Kullanılacak örnek sayıları
ada_times_list = []
xgb_times_list = []

for s in sizes:
    # Eğitim verisinin ilk s örneğini al
    Xs = X_train.iloc[:s]
    ys = y_train.iloc[:s]

    # AdaBoost süresi
    m1 = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, learning_rate=0.1, algorithm='SAMME', random_state=42
    )
    t1 = time.time()
    m1.fit(Xs, ys)
    ada_times_list.append(time.time() - t1)

    # XGBoost süresi
    m2 = xgb.XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        eval_metric='logloss', random_state=42, verbosity=0
    )
    t2 = time.time()
    m2.fit(Xs, ys)
    xgb_times_list.append(time.time() - t2)

plt.figure(figsize=(10, 5))
plt.plot(sizes, ada_times_list, 'r-o', markersize=8, linewidth=2, label='AdaBoost')
plt.plot(sizes, xgb_times_list, 'g-s', markersize=8, linewidth=2, label='XGBoost')
# Beklenti: XGBoost büyük veride paralel işlem sayesinde daha hızlı olabilir.

plt.xlabel('Eğitim Örnek Sayısı', fontsize=12)
plt.ylabel('Eğitim Süresi (sn)', fontsize=12)
plt.title('Eğitim Süresi Karşılaştırması', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nFull dataset eğitim süreleri:')
print(f'  🔴 AdaBoost : {ada_time:.4f} sn')
print(f'  🟢 XGBoost  : {xgb_time:.4f} sn')

---
## 📋 10. Nihai Sonuç Tablosu

In [ ]:
# ─── Tüm metrikleri tek tabloda özetle ──────────────────────────────────────
results_df = pd.DataFrame({
    'Metrik': ['Accuracy', 'AUC-ROC', 'F1 Score', 'Precision', 'Recall',
               'CV Ortalama', 'CV Std', 'Eğitim Süresi (sn)'],
    'AdaBoost 🔴': [
        f'{ada_acc:.4f}', f'{ada_auc:.4f}', f'{ada_f1:.4f}',
        f'{ada_prec:.4f}', f'{ada_recall:.4f}',
        f'{ada_cv.mean():.4f}', f'{ada_cv.std():.4f}',
        f'{ada_time:.4f}'
    ],
    'XGBoost 🟢': [
        f'{xgb_acc:.4f}', f'{xgb_auc:.4f}', f'{xgb_f1:.4f}',
        f'{xgb_prec:.4f}', f'{xgb_recall:.4f}',
        f'{xgb_cv.mean():.4f}', f'{xgb_cv.std():.4f}',
        f'{xgb_time:.4f}'
    ],
    'Kazanan': [
        # Her metrik için iki modeli karşılaştır ve kazananı belirle
        '🟢 XGBoost' if xgb_acc    >= ada_acc    else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_auc    >= ada_auc    else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_f1     >= ada_f1     else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_prec   >= ada_prec   else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_recall >= ada_recall else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_cv.mean() >= ada_cv.mean() else '🔴 AdaBoost',
        '🟢 XGBoost' if xgb_cv.std()  <= ada_cv.std()  else '🔴 AdaBoost',
        # Süre için: daha kısa süre kazanır
        '🟢 XGBoost' if xgb_time <= ada_time else '🔴 AdaBoost'
    ]
})

print('\n' + '='*65)
print('         📊 ADABOOST vs XGBOOST — SONUÇ TABLOSU')
print('='*65)
print(results_df.to_string(index=False))
print('='*65)

# Kazanılan kategori sayısını say
xgb_wins = sum(1 for k in results_df['Kazanan'] if 'XGBoost' in k)
ada_wins = sum(1 for k in results_df['Kazanan'] if 'AdaBoost' in k)
print(f'\n🏆 GENEL SONUÇ:')
print(f'   🟢 XGBoost kazandı  : {xgb_wins}/{len(results_df)} kategoride')
print(f'   🔴 AdaBoost kazandı : {ada_wins}/{len(results_df)} kategoride')

---

## 🎯 11. Hangi Algoritmayı Kullanmalısınız?

### ✅ AdaBoost Kullanın — Eğer:
- Verinin gürültüsü **düşük** ve temizse
- **Yorumlanabilirlik** önemli ise (karar stumps kolayca yorumlanır)
- Veri seti **küçük** ve hızlı prototip gerekiyorsa
- Hiperparametre ayarı için zamanınız **kısıtlıysa**

### ✅ XGBoost Kullanın — Eğer:
- **En yüksek doğruluğu** hedefliyorsanız
- Verinizde **eksik değerler (NaN)** varsa
- **Büyük veri setleri** ile çalışıyorsanız
- **Outlier'lar** ve gürültülü veri söz konusuysa
- **Kaggle gibi yarışmalarda** katılıyorsanız (XGBoost = defacto standart)
- Regularizasyon ile **overfit önlemek** istiyorsanız

---

## 🚀 12. Özet

| | AdaBoost | XGBoost |
|---|---|---|
| **Güçlü yönü** | Basitlik, hız | Performans, esneklik |
| **Zayıf yönü** | Outlier hassasiyeti | Karmaşık hiperparametre |
| **En iyi durum** | Küçük, temiz veri | Her tür veri |
| **Endüstri kullanımı** | Yüz tanıma (Viola-Jones) | Finance, sağlık, Kaggle |

> 💡 **Kural:** AdaBoost başlangıç noktanız olsun, XGBoost üretim modeliniz.